In [ ]:
import duckdb
import pandas as pd

In [ ]:
conn = duckdb.connect(database=':memory:')

In [ ]:
conn.execute("INSTALL tpch;")
conn.execute("LOAD tpch;")

In [ ]:
# For reference, lineitem with SF=1 ≈ 6 million rows.
# 0.34 * 6 million ≈ 2 million rows.
scale_factor = 0.34

In [ ]:
conn.execute(f"""
DROP TABLE IF EXISTS customer;
DROP TABLE IF EXISTS lineitem;
DROP TABLE IF EXISTS nation;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS part;
DROP TABLE IF EXISTS partsupp;
DROP TABLE IF EXISTS region;
DROP TABLE IF EXISTS supplier;
""")

In [ ]:
conn.execute(f"CALL dbgen(sf={scale_factor});")

In [ ]:
lineitem_count = conn.execute("SELECT COUNT(*) FROM lineitem;").fetchone()[0]
print(f"LINEITEM rows generated: {lineitem_count:,}")

In [ ]:
df_sample = conn.execute("SELECT * FROM lineitem LIMIT 5;").fetchdf()
display(df_sample)

In [ ]:
# We export to csv since the experiments require it.
# parquet files are generally faster, but aren't (still) supported
# by our benchmarking tool.
output_path = "../data/lineitem_2m.csv"
conn.execute(f"COPY lineitem TO '{output_path}' (HEADER, DELIMITER ',');")

In [ ]:
conn.execute(f"FROM tpch_queries();").fetchdf()